In [ ]:
# Vision Transformers and Scaling Laws
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part4/16-vit-scaling.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = [
    {
        "path": "data/fashion-test.pt",
        "sha256": "1db79d080c51173c7df18a5e8389dd4ae20ecb0352a21be90aaa446aed612a09"
    },
    {
        "path": "data/fashion-train.pt",
        "sha256": "86a99167f14d98891de2bc34b83197727f89e178cdf9e5b019fceb7bf71cc427"
    }
]

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part4').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part4')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Load imports, the fixed Chapter 6 split, and image shapes.

In [ ]:
import hashlib
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

# [1]
torch.set_num_threads(6)
torch.manual_seed(6050)

development = torch.load("../../data/fashion-train.pt")
holdout = torch.load("../../data/fashion-test.pt")
X_dev = development["X"].float().unsqueeze(1) / 255.0  # (1200, 1, 28, 28)
y_dev, classes = development["y"], development["classes"]
X_test = holdout["X"].float().unsqueeze(1) / 255.0     # (600, 1, 28, 28)
y_test = holdout["y"]

split = torch.randperm(len(X_dev), generator=torch.Generator().manual_seed(6050))
fit_idx, val_idx = split[:1000], split[1000:]
X_fit, y_fit = X_dev[fit_idx], y_dev[fit_idx]           # (1000, 1, 28, 28)
X_val, y_val = X_dev[val_idx], y_dev[val_idx]           # (200, 1, 28, 28)

# [2]
print("fit / validation / benchmark:", len(X_fit), len(X_val), len(X_test))

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Patchify and verify unfold-plus-linear equals one strided convolution.

In [ ]:
# [1]
P, d = 4, 32
sample = X_fit[:1]
generator = torch.Generator().manual_seed(6050)
E_patch = torch.randn(P * P, d, generator=generator)
b_patch = torch.randn(d, generator=generator)

raw_patches = F.unfold(sample, kernel_size=P, stride=P).transpose(1, 2)
tokens_linear = raw_patches @ E_patch + b_patch
conv_kernel = E_patch.T.reshape(d, 1, P, P)
tokens_conv = F.conv2d(
    sample, conv_kernel, bias=b_patch, stride=P
).flatten(2).transpose(1, 2)
max_error = (tokens_linear - tokens_conv).abs().max().item()

assert raw_patches.shape == (1, 49, 16)
assert tokens_linear.shape == (1, 49, 32)
assert max_error < 2e-6
# [2]
print("raw patches:", tuple(raw_patches.shape))
print("projected tokens:", tuple(tokens_linear.shape))
print(f"max |linear - convolution|: {max_error:.2e}")

**Plan**

1. Define the reusable helpers: `MultiHeadSelfAttention`, `EncoderBlock`, and `TinyViT`.
2. Define the reusable helpers: `TinyCNN`, `_xavier_init`, and `count_parameters`.
3. Independently define the tiny ViT and parameter-matched CNN.

In [ ]:
# [1]
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, width: int, heads: int) -> None:
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_width = width // heads
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, tokens, width = x.shape
        qkv = self.qkv(x).reshape(
            batch, tokens, 3, self.heads, self.head_width
        ).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_width)
        mixed = scores.softmax(dim=-1) @ v
        mixed = mixed.transpose(1, 2).reshape(batch, tokens, width)
        return self.proj(mixed)


class EncoderBlock(nn.Module):
    def __init__(self, width: int, heads: int, ff_width: int) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(width)
        self.attn = MultiHeadSelfAttention(width, heads)
        self.norm2 = nn.LayerNorm(width)
        self.ffn = nn.Sequential(
            nn.Linear(width, ff_width), nn.GELU(), nn.Linear(ff_width, width)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))


class TinyViT(nn.Module):
    def __init__(self, width: int = 32, patch: int = 4, blocks: int = 2) -> None:
        super().__init__()
        self.patch_embed = nn.Conv2d(1, width, kernel_size=patch, stride=patch)
        patch_tokens = (28 // patch) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, width))
        self.position = nn.Parameter(torch.zeros(1, patch_tokens + 1, width))
        self.blocks = nn.Sequential(*(
            EncoderBlock(width, heads=4, ff_width=2 * width)
            for _ in range(blocks)
        ))
        self.norm = nn.LayerNorm(width)
        self.head = nn.Linear(width, 10)
        self.apply(_xavier_init)
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.position, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        patches = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(len(x), -1, -1)
        tokens = torch.cat((cls, patches), dim=1) + self.position
        encoded = self.blocks(tokens)
        return self.head(self.norm(encoded[:, 0]))


# [2]
class TinyCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.GELU(),
            nn.Conv2d(32, 48, 3, padding=1), nn.GELU(),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(48, 10)
        )
        self.apply(_xavier_init)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


def _xavier_init(module: nn.Module) -> None:
    if isinstance(module, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


# [3]
print("CNN parameters:", f"{count_parameters(TinyCNN()):,}")
print("ViT parameters:", f"{count_parameters(TinyViT()):,}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable helpers: `minibatch_schedule`, `schedule_hash`, and `train_with_schedule`.
3. Define the reusable helpers: `accuracy`, `shift_right`, and `run_paired_seed`.
4. Define the explicit paired schedule, training loop, and shift audit.

In [ ]:
# [1]
EPOCHS = 120
SEEDS = list(range(6050, 6055))


# [2]
def minibatch_schedule(seed: int) -> list[torch.Tensor]:
    generator = torch.Generator().manual_seed(seed + 10_000)
    return [torch.randperm(len(X_fit), generator=generator) for _ in range(EPOCHS)]


def schedule_hash(permutations: list[torch.Tensor]) -> str:
    raw = torch.cat(permutations).numpy().tobytes()
    return hashlib.sha256(raw).hexdigest()[:12]


def train_with_schedule(
    model_fn, permutations: list[torch.Tensor], seed: int
) -> nn.Module:
    torch.manual_seed(seed)
    model = model_fn()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=3e-5
    )
    model.train()
    for permutation in permutations:
        for start in range(0, len(X_fit), 100):
            idx = permutation[start:start + 100]
            loss = F.cross_entropy(model(X_fit[idx]), y_fit[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
    return model


# [3]
@torch.no_grad()
def accuracy(model: nn.Module, x: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    return (model(x).argmax(1) == y).float().mean().item()


def shift_right(x: torch.Tensor, pixels: int) -> torch.Tensor:
    if pixels == 0:
        return x.clone()
    shifted = torch.zeros_like(x)
    shifted[..., pixels:] = x[..., :-pixels]
    return shifted


def run_paired_seed(seed: int) -> list[dict]:
    permutations = minibatch_schedule(seed)
    rows = []
    print(f"seed {seed}, schedule {schedule_hash(permutations)}")
    for name, model_fn in (("CNN", TinyCNN), ("ViT", TinyViT)):
        model = train_with_schedule(model_fn, permutations, seed)
        row = {
            "seed": seed,
            "model": name,
            "train": accuracy(model, X_fit, y_fit),
            "validation": [
                accuracy(model, shift_right(X_val, px), y_val) for px in range(5)
            ],
            "benchmark": accuracy(model, X_test, y_test),
        }
        rows.append(row)
        print(
            f"  {name}: train {row['train']:.3f}, "
            f"validation {[round(value, 3) for value in row['validation']]}, "
            f"benchmark {row['benchmark']:.3f}"
        )
    return rows


vit_macs = (
    7 * 7 * 32 * 16
    + 2 * (
        50 * 32 * (3 * 32) + 2 * 4 * 50 * 50 * 8
        + 50 * 32 * 32 + 2 * 50 * 32 * 64
    )
    + 32 * 10
)
cnn_macs = (
    28 * 28 * 8 * 1 * 3 * 3
    + 14 * 14 * 16 * 8 * 3 * 3
    + 7 * 7 * 32 * 16 * 3 * 3
    + 7 * 7 * 48 * 32 * 3 * 3
    + 48 * 10
)
# [4]
print(f"dot-product/MAC proxy — CNN: {cnn_macs:,}; ViT: {vit_macs:,}")

**Plan**

1. Paired Fashion run for seed 6050.

In [ ]:
# [1]
fashion_results = []
fashion_results.extend(run_paired_seed(6050))

**Plan**

1. Paired Fashion run for seed 6051.

In [ ]:
# [1]
fashion_results.extend(run_paired_seed(6051))

**Plan**

1. Paired Fashion run for seed 6052.

In [ ]:
# [1]
fashion_results.extend(run_paired_seed(6052))

**Plan**

1. Paired Fashion run for seed 6053.

In [ ]:
# [1]
fashion_results.extend(run_paired_seed(6053))

**Plan**

1. Paired Fashion run for seed 6054.

In [ ]:
# [1]
fashion_results.extend(run_paired_seed(6054))

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Report the paired clean and shifted endpoints.

In [ ]:
# [1]
by_model = {
    name: [row for row in fashion_results if row["model"] == name]
    for name in ("CNN", "ViT")
}

for name in by_model:
    assert [row["seed"] for row in by_model[name]] == SEEDS

colors = {"CNN": "#232D4B", "ViT": "#E57200"}
# [2]
for name in ("CNN", "ViT"):
    rows = by_model[name]
    mean_train = np.mean([row["train"] for row in rows])
    mean_val = np.mean([row["validation"][0] for row in rows])
    mean_test = np.mean([row["benchmark"] for row in rows])
    mean_shift4 = np.mean([row["validation"][4] for row in rows])
    print(
        f"{name}: train {mean_train:.1%}; clean validation {mean_val:.1%}; "
        f"benchmark {mean_test:.1%}; validation at 4px {mean_shift4:.1%}"
    )

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Audit the EfficientNet compound-scaling constraint.

In [ ]:
# [1]
alpha, beta, gamma = 1.2, 1.1, 1.15
phis = np.arange(5)
depth_multiplier = alpha ** phis
width_multiplier = beta ** phis
resolution_multiplier = gamma ** phis
compute_multiplier = (alpha * beta**2 * gamma**2) ** phis

# [2]
print(f"alpha * beta^2 * gamma^2 = {alpha * beta**2 * gamma**2:.5f}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Turn the published Kaplan exponents into auditable multipliers.

In [ ]:
# [1]
scales = np.logspace(0, 3, 200)
kaplan_exponents = {
    "parameters, $\\alpha=0.076$": (0.076, "#232D4B"),
    "tokens, $\\alpha=0.095$": (0.095, "#E57200"),
    "optimal compute, $\\alpha=0.050$": (0.050, "#2E7D32"),
}

# [2]
for label, (exponent, _) in kaplan_exponents.items():
    doubling_factor = 2 ** (-exponent)
    print(
        f"{label}: doubling leaves {doubling_factor:.4f} "
        f"({100 * (1 - doubling_factor):.1f}% reduction)"
    )

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable `fitted_chinchilla_loss` helper.
3. Report the joint-fit optimum beside the 20-token heuristic.

In [ ]:
# [1]
E_inf, A_fit, B_fit = 1.69, 406.4, 410.7
alpha_fit, beta_fit = 0.34, 0.28
C_budget = 5.76e23

G_fit = (
    alpha_fit * A_fit / (beta_fit * B_fit)
) ** (1 / (alpha_fit + beta_fit))
N_joint = G_fit * (C_budget / 6) ** (
    beta_fit / (alpha_fit + beta_fit)
)
D_joint = (C_budget / 6) / N_joint

N_twenty = math.sqrt(C_budget / 120)
D_twenty = 20 * N_twenty

N_grid = np.logspace(np.log10(4e9), np.log10(5e11), 400)
D_grid = C_budget / (6 * N_grid)
loss_grid = (
    E_inf + A_fit / N_grid**alpha_fit + B_fit / D_grid**beta_fit
)

# [2]
def fitted_chinchilla_loss(parameters: float, tokens: float) -> float:
    return (
        E_inf
        + A_fit / parameters**alpha_fit
        + B_fit / tokens**beta_fit
    )

points = [
    (N_joint, D_joint, "joint-fit minimum", "#2E7D32"),
    (N_twenty, D_twenty, "20-token heuristic", "#E57200"),
    (280e9, C_budget / (6 * 280e9), "Gopher-sized model", "#722F37"),
]

# [3]
print(
    f"joint-fit minimum: N={N_joint / 1e9:.2f}B, "
    f"D={D_joint / 1e12:.3f}T, D/N={D_joint / N_joint:.1f}"
)
print(
    f"20-token heuristic: N={N_twenty / 1e9:.2f}B, "
    f"D={D_twenty / 1e12:.3f}T, C={6 * N_twenty * D_twenty:.2e}"
)